✍ Для поиска необходимых нам данных мы будем использовать библиотеку BeautifulSoup, которая позволяет по названию тегов и их атрибутов получать содержащийся в них текст.

BeautifulSoup не является частью стандартной библиотеки, поэтому для начала её нужно установить. Например, в Jupyter Notebook это делается с помощью такой команды:

In [1]:
!pip install beautifulsoup4

In [2]:
# Импортируем библиотеку
from bs4 import BeautifulSoup

Теперь мы можем извлекать данные из любой веб-страницы.

Ранее мы уже получили содержимое страницы с помощью GET-запроса и сохранили информацию в переменной response , теперь создадим объект BeautifulSoup с именем page, указывая в качестве параметра html.parser.

Для примера получим информацию o title (с англ. заголовок) — это строка, которая отображается на вкладке браузера:

In [22]:
import requests

url = 'https://nplus1.ru/news/2021/10/11/econobel2021'
# Выполняем GET-запрос, содержимое ответа присваивается переменной response
response = requests.get(url)
# Создаём объект BeautifulSoup, указывая html-парсер
page = BeautifulSoup(response.text, 'html.parser')
# Получаем тег title, отображающийся на вкладке браузера
print(page.title)
# Выводим текст из полученного тега, который содержится в атрибуте text
print(page.title.text)

<title>Премию Нобеля по экономике присудили за исследования экономики труда и причинно-следственных связей</title>
Премию Нобеля по экономике присудили за исследования экономики труда и причинно-следственных связей


Если при запросе к сайту, а затем при его разборе с помощью BeautifulSoup в тексте страницы не находится нужный тег, попробуйте вывести на печать пару тысяч символов текста страницы. Если там обнаружится нечто похожее на капчу, возможно, сайт посчитал вас роботом и отказывается выдавать содержимое. Чтобы получить его, попробуйте «притвориться» браузером при запросе из скрипта:

In [23]:
requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})

<Response [200]>

Выполним поставленную ранее задачу: получить информацию о странице и извлечь заголовок статьи, опубликованной на этой странице, дату публикации, а также текст статьи.

Предположим, что мы знаем, что в HTML-коде рассматриваемой нами страницы заголовок статьи заключён в тег h1 (заголовок первого уровня).

Тогда мы можем получить его текст с помощью метода find() (с англ. найти) объекта BeautifulSoup, передав ему название интересующего нас тега:

In [24]:
# Применяем метод find() к объекту и выводим результат на экран
print(page.find('h1').text)


            Премию Нобеля по экономике присудили за исследования экономики труда и причинно-следственных связей
          


Но как же узнать, в каких именно тегах заключена необходимая информация?

Проще всего это сделать с помощью так называемого инструмента разработчика, который есть во всех современных браузерах. Покажем, как открыть данный инструмент на примере использования браузера Google Chrome.

Устанавливаем курсор на элементе страницы (заголовок статьи), информацию о котором хотим получить, нажимаем на правую клавишу мыши и в выпадающем списке выбираем пункт «Просмотреть код элемента» или «Исследовать» в зависимости от браузера.

В открывшемся окне инструмента разработчика видим, что информация о заголовке статьи заключена в теге h1.

Напишите функцию wiki_header, которая по адресу страницы возвращает заголовок первого уровня для статей на Wikipedia.

Функция wiki_header принимает один аргумент - url.

wiki_header('https://en.wikipedia.org/wiki/Operating_system')

'Operating system'

In [20]:
def wiki_header(url):
    # Обязательно добавляем User-Agent, чтобы не вернулся None и не вышла ошибка
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    page = BeautifulSoup(response.text, 'html.parser')
    header = page.find('h1').text
    return header
    
wiki_header('https://en.wikipedia.org/wiki/Operating_system')

'Operating system'

НЕУНИКАЛЬНЫЕ ТЕГИ: ИЗВЛЕКАЕМ ТЕКСТ И ДАТУ ПУБЛИКАЦИИ СТАТЬИ

Теперь получим сам текст статьи. Как вы уже знаете, первым делом необходимо определить, в какой тег он заключён. Применим, как и ранее, инструмент разработчика.

Видим, что искомый текст заключён в тег div. Попробуем извлечь его уже известным нам способом — с помощью метода find() — и выведем его на экран.

In [25]:
print(page.find('div').text)

Мы увидели не то, что ожидали — кучу текста, не имеющего отношения к тому, что мы искали...

В чём же проблема?

Дело в том, что теги div очень распространённые и на странице их очень много. Метод find() нашёл первый из них, но это не то, что нам надо.

Посмотрим на нашу страницу, используя инструмент разработчика, ещё раз. Можем заметить, что у искомого текста есть свой класс — n1_material text-18.

Передадим название класса в метод find() с помощью аргумента class_ и получим текст статьи:

In [27]:
print(page.find('div', class_='n1_material text-18').text)

Премия Шведского национального банка по экономическим наукам памяти Альфреда Нобеля за 2021 год присуждена Дэвиду Карду (David Card) за его вклад в эмпирические исследования экономики рынка труда, а также Джошуа Энгристу (Joshua Angrist) и Гвидо Имбенсу (Guido Imbens) за их вклад в методологию анализа причинно-следственных связей. Прямая трансляция церемонии объявления лауреатов шла на официальном сайте Нобелевской премии.


В данном случае происходит поиск точного строкового значения class атрибута, т. е. выполнение строк кода даст одинаковый результат:

In [28]:
print(page.find('div', class_='n1_material').text)
print(page.find('div', class_='n1_material text-18').text)

Премия Шведского национального банка по экономическим наукам памяти Альфреда Нобеля за 2021 год присуждена Дэвиду Карду (David Card) за его вклад в эмпирические исследования экономики рынка труда, а также Джошуа Энгристу (Joshua Angrist) и Гвидо Имбенсу (Guido Imbens) за их вклад в методологию анализа причинно-следственных связей. Прямая трансляция церемонии объявления лауреатов шла на официальном сайте Нобелевской премии.
Премия Шведского национального банка по экономическим наукам памяти Альфреда Нобеля за 2021 год присуждена Дэвиду Карду (David Card) за его вклад в эмпирические исследования экономики рынка труда, а также Джошуа Энгристу (Joshua Angrist) и Гвидо Имбенсу (Guido Imbens) за их вклад в методологию анализа причинно-следственных связей. Прямая трансляция церемонии объявления лауреатов шла на официальном сайте Нобелевской премии.


А при выполнении этой строки кода мы получим ошибку, так как такого строкового значения в области поиска нет.

In [29]:
print(page.find('div', class_='text-18 n1_material').text)

AttributeError: 'NoneType' object has no attribute 'text'

Аналогично получим информации о теге, который содержит дату написания статьи, отображаемую в левом верхнем углу страницы.

Итак, нам нужен тег **"a"** с классом "relative before:block before:w-px before:bg-current before:h-4 before:absolute before:left-0 group pl-2 flex inline-flex items-center". Для поиска достаточно указать в качестве класса "relative", отбросив дополнительные настройки.

Теперь получим данные из него с помощью уже известного метода find(), передав название нужного тега:

In [ ]:
# Выводим на экран содержимое атрибута text тега a с классом "relative"
print(page.find('a', class_='relative').text)


11.10.21



СБОР НЕСКОЛЬКИХ ЭЛЕМЕНТОВ: СОБИРАЕМ ВСЕ ССЫЛКИ НА СТРАНИЦЕ

Рассмотрим ещё один сценарий: вы хотите собрать сразу несколько элементов со страницы. Например, представьте, что вы хотите получить названия всех языков программирования, упомянутых на странице в Wikipedia в статье про языки программирования.

Можно заметить, что все названия языков программирования на этой странице связаны ссылками c соответствующими статьями о них. Таким образом, нам необходимо собрать все ссылки на странице. Для ссылок в HTML предусмотрен тег **"a"**. Попробуем использовать find():

In [35]:
url = 'https://en.wikipedia.org/wiki/List_of_programming_languages'

response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
page = BeautifulSoup(response.text, 'html.parser')

print(page.find('a'))

<a class="mw-jump-link" href="#bodyContent">Jump to content</a>


Мы получили только одну ссылку, хотя на странице их явно больше.

Это происходит, потому что метод find() возвращает только первый подходящий элемент. Если требуется получить больше элементов, необходимо воспользоваться методом find_all() (с англ. найти все):

In [37]:
links = page.find_all('a')
print(len(links))

966


Посмотрим на некоторые ссылки:

In [39]:
print([link.text for link in links[500:510]]) # Выводим ссылки с 500 по 509 включительно

['Magma', 'Maple', 'MAPPER', 'MARK-IV', 'Mary', 'MASM Microsoft Assembly x86', 'MATLAB', 'MATH-MATIC', 'Maude system', 'Max']


Не все ссылки соответствуют названиям языков программирования — страница содержит также «служебные» ссылки, такие, например, как Jump to navigation (с англ. Перейти к навигации) или Alphabetical (с англ. По алфавиту):

In [45]:
print([link.text for link in links[0:150]]) # Выводим ссылки с 1 по 9 включительно

['Jump to content', 'Main page', 'Contents', 'Current events', 'Random article', 'About Wikipedia', 'Contact us', 'Help', 'Learn to edit', 'Community portal', 'Recent changes', 'Upload file', 'Special pages', '\n\n\n\n\n\n', '\nSearch\n', 'Donate', 'Create account', 'Log in', 'Donate', ' Create account', ' Log in', 'Afrikaans', 'العربية', 'Azərbaycanca', 'Български', 'বাংলা', 'Bosanski', 'Basa Ugi', 'Català', 'Čeština', 'Deutsch', 'Esperanto', 'Español', 'Euskara', 'فارسی', 'Suomi', 'Français', 'हिन्दी', 'Hrvatski', 'Kreyòl ayisyen', 'Magyar', 'Հայերեն', 'Bahasa Indonesia', 'Íslenska', 'Italiano', '日本語', '한국어', 'Lëtzebuergesch', 'Latviešu', 'Монгол', 'Bahasa Melayu', 'Nederlands', 'Norsk nynorsk', 'Norsk bokmål', 'Polski', 'Português', 'Română', 'Srpskohrvatski / српскохрватски', 'Simple English', 'Slovenčina', 'Slovenščina', 'Shqip', 'Српски / srpski', 'Sunda', 'தமிழ்', 'Тоҷикӣ', 'Türkçe', 'Українська', 'Oʻzbekcha / ўзбекча', 'Tiếng Việt', '粵語', '中文', 'Edit links', 'Article', 'Talk', 

Для обработки полученных данных и исключения «лишней» информации можно, например, использовать подходы, которые вы изучили в модуле PY-14 Очистка данных.